**When Is TSLS Actually LATE? Evidence from Project STAR**

Anastasiia Rekovets

---
Replication of Krueger (1999) evaluating the LATE interpretation of TSLS under the framework of Blandhol et al. (2022).

**Data:** Place `Cleandata.dta` in `./data/` before running.

## 0. Setup

In [1]:
# Uncomment to install if needed:
# !pip install linearmodels DoubleML

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import reset_ramsey
from linearmodels.iv import IV2SLS
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from doubleml import DoubleMLData, DoubleMLPLR, DoubleMLIIVM
import doubleml as dml
from IPython.display import display

## 1. Data Loading and Initial Preparation

In [2]:
df = pd.read_stata('./data/Cleandata.dta')
data = df.copy()
data.head()

,stdntid,gender,race,birthmonth,birthday,birthyear,flagsgk,flagsg1,flagsg2,flagsg3,...,g2s,g2r,g2ra,g3s,g3r,g3ra,fyclasstype,initials,initialr,initialra
0,11465,male,black,march,13.0,1979.0,no,no,no,yes,...,NaN,NaN,NaN,0.0,1.0,0.0,2.0,0.0,1.0,0.0
1,11047,male,black,august,30.0,1978.0,no,no,no,yes,...,NaN,NaN,NaN,0.0,0.0,1.0,3.0,0.0,0.0,1.0
2,20757,male,white,october,22.0,1979.0,no,no,no,yes,...,NaN,NaN,NaN,0.0,0.0,1.0,3.0,0.0,0.0,1.0
3,12959,male,black,july,25.0,1979.0,no,yes,yes,yes,...,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,0.0
4,19191,female,black,arpil,28.0,1979.0,no,no,yes,yes,...,0.0,0.0,1.0,0.0,0.0,1.0,3.0,0.0,0.0,1.0


In [3]:
# Create initial assignment groups from fyclasstype
df['initial_small']   = (df['fyclasstype'] == 1).astype(int)
df['initial_regular'] = (df['fyclasstype'] == 2).astype(int)
df['initial_ra']      = (df['fyclasstype'] == 3).astype(int)

print('Initial assignment distribution:')
print(f"  Small:        {df['initial_small'].sum()}   ({df['initial_small'].mean()*100:.1f}%)")
print(f"  Regular:      {df['initial_regular'].sum()}  ({df['initial_regular'].mean()*100:.1f}%)")
print(f"  Regular/Aide: {df['initial_ra'].sum()}   ({df['initial_ra'].mean()*100:.1f}%)")

Initial assignment distribution:
  Small:        3024   (26.1%)
  Regular:      4328  (37.3%)
  Regular/Aide: 4249   (36.6%)


## 2. Variable Construction

In [4]:
# ── Gender ────────────────────────────────────────────────────────────────────
df['girl'] = df['gender'].str.lower().map({'male': 0, 'female': 1}).fillna(0).astype(int)

# ── Race/Ethnicity ────────────────────────────────────────────────────────────
df['white_asian'] = df['WhiteAsian'].fillna(0).astype(int)

# ── Age as of September 1, 1985 ───────────────────────────────────────────────
df['birth_day']  = pd.to_numeric(df['birthday'],  errors='coerce')
df['birth_year'] = pd.to_numeric(df['birthyear'], errors='coerce')

month_map = {
    'january': 1, 'february': 2, 'march': 3,    'april':    4,
    'may':     5, 'june':     6, 'july':  7,    'august':   8,
    'september': 9, 'october': 10, 'november': 11, 'december': 12
}
birth_month_num = df['birthmonth'].str.lower().map(month_map)

df['age_1985'] = 1985 - df['birth_year']
born_after_sept_1st = (birth_month_num > 9) | ((birth_month_num == 9) & (df['birth_day'] > 1))
df.loc[born_after_sept_1st, 'age_1985'] -= 1

In [5]:
# ── Free lunch by grade ───────────────────────────────────────────────────────
for grade in ['0', '1', '2', '3']:
    col = f'g{grade}freelunch_2'
    if col in df.columns:
        df[f'free_lunch_{grade}'] = df[col].fillna(0).astype(int)

# ── Numeric test scores ───────────────────────────────────────────────────────
for grade in ['0', '1', '2', '3']:
    for subject in ['treadss', 'tmathss', 'wordskillss']:
        col = f'g{grade}{subject}'
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# ── Numeric class type ────────────────────────────────────────────────────────
class_mapping = {
    'SMALL CLASS': 1, 'REGULAR CLASS': 2,
    'REGULAR + AIDE CLASS': 3, 'REGULAR+AIDE CLASS': 3,
    'SMALL': 1, 'REGULAR': 2, 'REGULAR+AIDE': 3
}
for grade in ['0', '1', '2', '3']:
    col = f'g{grade}classtype'
    if col in df.columns:
        df[col] = df[col].astype(str).map(class_mapping)

In [6]:
# ── Percentile scores (relative to control group within each grade) ───────────
def calculate_percentile_score(score, control_group_scores):
    """Percentile rank relative to the control group."""
    if pd.isna(score):
        return np.nan
    return (np.sum(control_group_scores < score) / len(control_group_scores)) * 100

for grade in ['0', '1', '2', '3']:
    test_cols = [f'g{grade}treadss', f'g{grade}tmathss', f'g{grade}wordskillss']
    if all(col in df.columns for col in test_cols):
        df[f'sat_avg_{grade}'] = df[test_cols].mean(axis=1)

        class_col = f'g{grade}classtype'
        if class_col in df.columns:
            control_mask   = df[class_col].isin([2, 3])
            control_scores = df.loc[control_mask, f'sat_avg_{grade}'].dropna()

            if len(control_scores) > 0:
                df[f'percentile_{grade}'] = df[f'sat_avg_{grade}'].apply(
                    lambda x: calculate_percentile_score(x, control_scores)
                )
                ctrl_mean = df.loc[control_mask, f'percentile_{grade}'].mean()
                print(f"  Grade {grade}: Control N={len(control_scores)}, Mean percentile={ctrl_mean:.1f}")

  Grade 0: Control N=4130, Mean percentile=49.8
  Grade 1: Control N=4758, Mean percentile=49.9
  Grade 2: Control N=4481, Mean percentile=49.9
  Grade 3: Control N=4289, Mean percentile=49.9


In [7]:
# ── Entry grade ───────────────────────────────────────────────────────────────
def get_entry_grade(row):
    """First grade with both a class-type and a valid test score."""
    for grade in ['0', '1', '2', '3']:
        if pd.notna(row.get(f'g{grade}classtype')) and pd.notna(row.get(f'g{grade}treadss')):
            return grade
    return np.nan

df['entry_grade'] = df.apply(get_entry_grade, axis=1)

print('Entry grade distribution:')
entry_counts = df['entry_grade'].value_counts().sort_index()
grade_names  = {'0': 'Kindergarten', '1': 'Grade 1', '2': 'Grade 2', '3': 'Grade 3'}
for grade, count in entry_counts.items():
    print(f"  {grade_names.get(grade, grade)}: {count} ({count/len(df)*100:.1f}%)")

Entry grade distribution:
  Kindergarten: 5789 (49.9%)
  Grade 1: 2385 (20.6%)
  Grade 2: 1346 (11.6%)
  Grade 3: 1140 (9.8%)


## 3. Descriptive Statistics (Krueger 1999, Table I)

In [8]:
cohorts = {
    '0': {'title': 'A. Students who entered STAR in kindergarten', 'desc': 'kindergarten'},
    '1': {'title': 'B. Students who entered STAR in first grade',  'desc': 'first grade'},
    '2': {'title': 'C. Students who entered STAR in second grade', 'desc': 'second grade'},
    '3': {'title': 'D. Students who entered STAR in third grade',  'desc': 'third grade'}
}

variables = [
    ('free_lunch',  'free_lunch_{g}',  '1. Free lunch',                  '{:.2f}'),
    ('white_asian', 'white_asian',      '2. White/Asian',                 '{:.2f}'),
    ('age',         'age_1985',         '3. Age in 1985',                 '{:.2f}'),
    ('class_size',  'g{g}classsize',    '5. Class size in {desc}',        '{:.1f}'),
    ('percentile',  'percentile_{g}',   '6. Percentile score in {desc}',  '{:.1f}')
]

for grade_id, cohort_info in cohorts.items():
    print(f"\n{cohort_info['title']}")
    print('-' * 75)

    cohort_data = df[df['entry_grade'] == grade_id]
    if cohort_data.empty:
        print('No data available'); continue

    groups = {
        'Small':        cohort_data[cohort_data['initial_small']   == 1],
        'Regular':      cohort_data[cohort_data['initial_regular'] == 1],
        'Regular/Aide': cohort_data[cohort_data['initial_ra']      == 1]
    }

    print(f"{'Variable':<35} {'Small':<12} {'Regular':<12} {'Regular/Aide':<14} {'Joint P-Value'}")
    print('-' * 75)

    for var_key, col_pattern, display_template, format_str in variables:
        col_name     = col_pattern.format(g=grade_id)
        display_name = display_template.format(desc=cohort_info['desc'])

        means, anova_data = [], []
        for group_name in ['Small', 'Regular', 'Regular/Aide']:
            group_df = groups[group_name]
            if col_name in group_df.columns:
                valid_data = group_df[col_name].dropna()
                if not valid_data.empty:
                    means.append(format_str.format(valid_data.mean()))
                    anova_data.append(valid_data)
                else:
                    means.append('NA')
            else:
                means.append('NA')

        p_value_str = 'NA'
        if var_key != 'class_size' and len(anova_data) >= 2:
            f_stat, p_val = stats.f_oneway(*anova_data)
            if not np.isnan(p_val):
                p_value_str = '.00' if p_val < 0.01 else f'{p_val:.2f}'

        if len(means) == 3:
            print(f'{display_name:<35} {means[0]:<12} {means[1]:<12} {means[2]:<14} {p_value_str}')

    n_obs = [str(len(groups[g])) for g in ['Small', 'Regular', 'Regular/Aide']]
    print(f"{'N observations':<35} {n_obs[0]:<12} {n_obs[1]:<12} {n_obs[2]}")


A. Students who entered STAR in kindergarten
---------------------------------------------------------------------------
Variable                            Small        Regular      Regular/Aide   Joint P-Value
---------------------------------------------------------------------------
1. Free lunch                       0.47         0.47         0.50           0.19
2. White/Asian                      0.68         0.68         0.67           0.39
3. Age in 1985                      4.96         4.95         4.95           0.82
5. Class size in kindergarten       15.1         22.3         22.7           NA
6. Percentile score in kindergarten 55.2         49.8         49.8           .00
N observations                      1739         2006         2044

B. Students who entered STAR in first grade
---------------------------------------------------------------------------
Variable                            Small        Regular      Regular/Aide   Joint P-Value
-------------------------

## 4. Long-Format Panel Construction

In [9]:
long_data = []
records   = df.to_dict('records')

for idx, row in enumerate(records):
    for grade in ['0', '1', '2', '3']:
        percentile_col = f'percentile_{grade}'
        if not pd.notna(row.get(percentile_col)):
            continue

        class_type_val = row.get(f'g{grade}classtype')
        long_data.append({
            'student_id':   idx,
            'grade':        int(grade),
            'percentile':   row[percentile_col],
            # Actual class assignment
            'small_class':    int(class_type_val == 1) if pd.notna(class_type_val) else 0,
            'reg_aide_class': int(class_type_val == 3) if pd.notna(class_type_val) else 0,
            # Instruments (initial assignment)
            'initial_small': int(row.get('initial_small', 0)),
            'initial_ra':    int(row.get('initial_ra',    0)),
            # Student controls
            'white_asian': int(row.get('white_asian', 0)),
            'girl':        int(row.get('girl',        0)),
            'free_lunch':  int(row.get(f'free_lunch_{grade}', 0)),
            # School FE
            'school_id': str(row.get(f'g{grade}schid', '')),
            # Teacher controls
            'teacher_white':      row.get(f'WhiteTeacher{grade}', 0),
            'teacher_exp':        float(row[f'g{grade}tyears']) if pd.notna(row.get(f'g{grade}tyears')) else np.nan,
            'teacher_degree_raw': row.get(f'g{grade}thighdegree', ''),
            'class_size':         float(row.get(f'g{grade}classsize', np.nan))
        })

long_df = pd.DataFrame(long_data)

# Teacher degree → binary (Master's or above)
degree_map = {
    'bachelors': 0, 'specialist': 0,
    'masters': 1, 'masters +': 1, 'doctoral': 1
}
long_df['teacher_masters'] = long_df['teacher_degree_raw'].astype(str).str.lower().map(degree_map)
long_df = long_df.drop(columns=['teacher_degree_raw'])

# Derive initial_regular
long_df['initial_regular'] = (1 - long_df['initial_small'] - long_df['initial_ra']).clip(lower=0)

long_df.head()

,student_id,grade,percentile,small_class,reg_aide_class,initial_small,initial_ra,white_asian,girl,free_lunch,school_id,teacher_white,teacher_exp,class_size,teacher_masters,initial_regular
0,0,3,0.000000,0,0,0,0,0,0,1,169280.0,1.0,30.0,25.0,0.0,1
1,1,3,0.046631,0,1,0,1,0,0,1,193422.0,1.0,17.0,25.0,1.0,0
2,2,3,0.279785,0,1,0,1,1,0,0,264945.0,1.0,16.0,28.0,1.0,0
3,3,1,6.809584,0,0,0,0,0,0,1,244780.0,1.0,2.0,25.0,0.0,1
4,3,2,4.909618,0,0,0,0,0,0,1,244780.0,1.0,1.0,21.0,0.0,1


## 5. OLS and Reduced-Form Regressions

In [10]:
def run_regression_extended(data, grade, model_type, controls_level):
    """
    OLS (model_type='ols') or Reduced Form (model_type='reduced_form').
    controls_level: 1=none, 2=+school FE, 3=+student, 4=+teacher.
    SEs clustered at school level.
    """
    df_g = data[data['grade'] == grade].copy()

    treatment_vars = (
        ['small_class', 'reg_aide_class'] if model_type == 'ols'
        else ['initial_small', 'initial_ra']
    )
    student_controls = ['white_asian', 'girl', 'free_lunch']
    teacher_controls = ['teacher_white', 'teacher_exp', 'teacher_masters']

    controls      = []
    use_school_fe = False
    if controls_level >= 2: use_school_fe = True
    if controls_level >= 3: controls.extend(student_controls)
    if controls_level >= 4: controls.extend(teacher_controls)

    req_cols = ['percentile'] + treatment_vars + controls
    if use_school_fe:
        req_cols.append('school_id')
    req_cols = [c for c in req_cols if c in df_g.columns]
    df_reg   = df_g[req_cols].dropna().copy()

    if len(df_reg) < 10:
        return None

    y = df_reg['percentile'].astype(float)
    X = df_reg[treatment_vars + controls].astype(float)
    X = sm.add_constant(X)

    if use_school_fe:
        school_dummies = pd.get_dummies(
            df_reg['school_id'].astype(str), prefix='sch', drop_first=True, dtype=float
        )
        X = pd.concat([X, school_dummies], axis=1)

    try:
        model = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': df_reg['school_id']})
    except Exception:
        try:
            model = sm.OLS(y, X).fit(cov_type='HC1')
        except Exception:
            model = sm.OLS(y, X).fit()

    return {
        'params':       model.params.to_dict(),
        'bse':          model.bse.to_dict(),
        'r2':           model.rsquared,
        'n_obs':        len(df_reg),
        'school_fe':    use_school_fe,
        'controls_level': controls_level,
        'model_type':   model_type
    }


def format_coef_se(coef, se, add_stars=False):
    """Format coefficient with SE and optional significance stars."""
    if coef is None or se is None or pd.isna(coef):
        return '—'
    stars = ''
    if add_stars and se > 0:
        t = abs(coef / se)
        stars = '***' if t > 2.576 else '**' if t > 1.960 else '*' if t > 1.645 else ''
    decimals = 3 if abs(se) < 0.01 else 2
    return f'{coef:.{decimals}f}{stars} ({se:.{decimals}f})'

In [11]:
results = {}
grades      = [0, 1, 2, 3]
grade_names = ['Kindergarten', 'Grade 1', 'Grade 2', 'Grade 3']

for grade_num, grade_name in zip(grades, grade_names):
    grade_results = {}
    for col in [1, 2, 3, 4]:
        grade_results[f'ols_{col}'] = run_regression_extended(long_df, grade_num, 'ols',          col)
        grade_results[f'rf_{col}']  = run_regression_extended(long_df, grade_num, 'reduced_form', col)
    results[grade_name] = grade_results

# Summary — specs (3) and (4) for each estimator
print('\n' + '='*100)
print('SUMMARY: SMALL CLASS EFFECTS — OLS AND REDUCED FORM')
print('='*100)
summary_headers = ['Grade', 'OLS (3)', 'OLS (4)', 'Reduced Form (7)', 'Reduced Form (8)']
print(' | '.join([h.ljust(18) for h in summary_headers]))
print('-'*100)

for grade_name in grade_names:
    grade_res = results[grade_name]
    row_vals  = [grade_name]
    for spec, var_name in [('ols_3', 'small_class'), ('ols_4', 'small_class'),
                            ('rf_3', 'initial_small'), ('rf_4', 'initial_small')]:
        res = grade_res.get(spec)
        if res:
            row_vals.append(format_coef_se(res['params'].get(var_name), res['bse'].get(var_name), add_stars=True))
        else:
            row_vals.append('—')
    print(' | '.join([str(x).ljust(18) for x in row_vals]))

print('\nNote: *** p<0.01, ** p<0.05, * p<0.10')


SUMMARY: SMALL CLASS EFFECTS — OLS AND REDUCED FORM
Grade              | OLS (3)            | OLS (4)            | Reduced Form (7)   | Reduced Form (8)  
----------------------------------------------------------------------------------------------------
Kindergarten       | 5.79*** (1.45)     | 5.66*** (1.42)     | 5.79*** (1.45)     | 5.66*** (1.42)    
Grade 1            | 8.63*** (1.43)     | 8.28*** (1.45)     | 7.32*** (1.32)     | 7.10*** (1.31)    
Grade 2            | 6.26*** (1.51)     | 6.28*** (1.51)     | 5.96*** (1.31)     | 5.91*** (1.31)    
Grade 3            | 5.50*** (1.32)     | 5.37*** (1.34)     | 5.86*** (1.14)     | 5.79*** (1.19)    

Note: *** p<0.01, ** p<0.05, * p<0.10


## 6. 2SLS — With Covariates (Table VII)

In [12]:
def run_2sls_regression(data, grade, include_controls=True, include_school_fe=True):
    """
    2SLS via linearmodels.IV2SLS.
    Endogenous: small_class. Instruments: initial_small, initial_regular.
    """
    df_g = data[data['grade'] == grade].copy()
    req  = ['percentile', 'small_class', 'initial_small', 'initial_regular']
    df_g = df_g.dropna(subset=req)
    if len(df_g) < 10:
        return None

    exog_cols = []
    if include_controls:
        df_g['teacher_exp'] = df_g['teacher_exp'].fillna(df_g['teacher_exp'].median())
        for col in ['white_asian', 'girl', 'free_lunch',
                    'teacher_white', 'teacher_exp', 'teacher_masters']:
            if col in df_g.columns:
                df_g[col] = df_g[col].fillna(df_g[col].mode()[0])
                exog_cols.append(col)

    if include_school_fe:
        school_dummies = pd.get_dummies(
            df_g['school_id'].astype(str), prefix='sch', drop_first=True, dtype=float
        )
        df_g       = pd.concat([df_g, school_dummies], axis=1)
        exog_cols += list(school_dummies.columns)

    y      = df_g['percentile']
    D      = df_g[['small_class']]
    Z      = df_g[['initial_small', 'initial_regular']]
    X_exog = (
        sm.add_constant(df_g[exog_cols].astype(float)) if exog_cols
        else sm.add_constant(pd.DataFrame(np.ones(len(df_g)), index=df_g.index, columns=['const']))
    )

    # First-stage F
    fs_f_stat = sm.OLS(D, pd.concat([Z, X_exog], axis=1)).fit().fvalue

    # OLS benchmark
    ols_model = sm.OLS(y, pd.concat([D, X_exog], axis=1)).fit()

    # 2SLS
    try:
        iv_model = IV2SLS(dependent=y, exog=X_exog, endog=D, instruments=Z).fit(cov_type='robust')
    except Exception as e:
        print(f'  Grade {grade} IV2SLS Error: {e}')
        return None

    return {
        'n_obs':    len(df_g),
        'fs_f_stat': fs_f_stat,
        'ols_coef': ols_model.params['small_class'],
        'ols_se':   ols_model.bse['small_class'],
        'iv_coef':  iv_model.params['small_class'],
        'iv_se':    iv_model.std_errors['small_class']
    }

In [13]:
table_rows = [['Grade', 'OLS', '2SLS', 'Sample Size', 'First Stage F']]

for grade_num, grade_name in zip(grades, grade_names):
    res = run_2sls_regression(long_df, grade_num, include_controls=True, include_school_fe=True)
    if res:
        table_rows.append([
            grade_name,
            f"{res['ols_coef']:.2f} ({res['ols_se']:.2f})",
            f"{res['iv_coef']:.2f} ({res['iv_se']:.2f})",
            f"{res['n_obs']:,}",
            f"{res['fs_f_stat']:.1f}"
        ])
    else:
        table_rows.append([grade_name, '—', '—', '—', '—'])

print('\n' + '='*100)
print('TABLE VII — OLS AND 2SLS ESTIMATES OF EFFECT OF SMALL CLASS ON ACHIEVEMENT')
print('DEPENDENT VARIABLE: AVERAGE PERCENTILE SCORE ON SAT')
print('='*100)
for row in table_rows:
    print(f'{row[0]:<15} {row[1]:<20} {row[2]:<20} {row[3]:<15} {row[4]}')
print('-'*90)
print('Note: School FE, student race, gender, free lunch, teacher race, experience,')
print('      and education included. Standard errors in parentheses.')


TABLE VII — OLS AND 2SLS ESTIMATES OF EFFECT OF SMALL CLASS ON ACHIEVEMENT
DEPENDENT VARIABLE: AVERAGE PERCENTILE SCORE ON SAT
Grade           OLS                  2SLS                 Sample Size     First Stage F
Kindergarten    5.55 (0.71)          5.55 (0.72)          5,902           38844994869215571783654047744.0
Grade 1         7.39 (0.68)          7.10 (0.79)          6,635           267.3
Grade 2         5.54 (0.70)          6.37 (0.85)          6,360           188.0
Grade 3         5.85 (0.72)          7.31 (0.94)          6,355           127.9
------------------------------------------------------------------------------------------
Note: School FE, student race, gender, free lunch, teacher race, experience,
      and education included. Standard errors in parentheses.


## 7. 2SLS — Without Covariates

In [14]:
def run_iv_no_covariates(data, grade):
    """
    Baseline IV with no controls and no school FE.
    Instrument: initial_small only.
    """
    df_g = data[data['grade'] == grade].copy()
    df_g = df_g.dropna(subset=['percentile', 'small_class', 'initial_small'])
    if len(df_g) < 10:
        return None

    y      = df_g['percentile']
    D      = df_g[['small_class']]
    Z      = df_g[['initial_small']]
    X_exog = sm.add_constant(
        pd.DataFrame(np.ones(len(df_g)), index=df_g.index, columns=['const'])
    )

    fs_f_stat = sm.OLS(D, pd.concat([Z, X_exog], axis=1)).fit().fvalue
    ols_model = sm.OLS(y, pd.concat([D, X_exog], axis=1)).fit()

    try:
        iv_model = IV2SLS(dependent=y, exog=X_exog, endog=D, instruments=Z).fit(cov_type='robust')
    except Exception as e:
        print(f'  Grade {grade} IV2SLS Error: {e}')
        return None

    return {
        'n_obs':     len(df_g),
        'fs_f_stat': fs_f_stat,
        'ols_coef':  ols_model.params['small_class'],
        'ols_se':    ols_model.bse['small_class'],
        'iv_coef':   iv_model.params['small_class'],
        'iv_se':     iv_model.std_errors['small_class']
    }


table_rows_nc = [['Grade', 'OLS', 'IV', 'Sample Size', 'First Stage F']]

for grade_num, grade_name in zip(grades, grade_names):
    res = run_iv_no_covariates(long_df, grade_num)
    if res:
        table_rows_nc.append([
            grade_name,
            f"{res['ols_coef']:.2f} ({res['ols_se']:.2f})",
            f"{res['iv_coef']:.2f} ({res['iv_se']:.2f})",
            f"{res['n_obs']:,}",
            f"{res['fs_f_stat']:.1f}"
        ])
    else:
        table_rows_nc.append([grade_name, '—', '—', '—', '—'])

print('\n' + '='*100)
print('IV ESTIMATES — NO COVARIATES')
print('DEPENDENT VARIABLE: AVERAGE PERCENTILE SCORE ON SAT')
print('='*100)
for row in table_rows_nc:
    print(f'{row[0]:<15} {row[1]:<20} {row[2]:<20} {row[3]:<15} {row[4]}')
print('-'*90)
print('Note: No covariates and no school fixed effects. Robust SEs in parentheses.')


IV ESTIMATES — NO COVARIATES
DEPENDENT VARIABLE: AVERAGE PERCENTILE SCORE ON SAT
Grade           OLS                  IV                   Sample Size     First Stage F
Kindergarten    5.06 (0.83)          5.06 (0.83)          5,902           631001830160937933183781676515328.0
Grade 1         7.67 (0.79)          8.10 (0.91)          6,635           20307.7
Grade 2         5.64 (0.79)          6.92 (0.96)          6,360           13657.7
Grade 3         6.13 (0.78)          7.90 (1.01)          6,355           9128.3
------------------------------------------------------------------------------------------
Note: No covariates and no school fixed effects. Robust SEs in parentheses.


## 8. Ramsey RESET Test for the Rich Covariates Condition

Following Blandhol et al. (2022): rejection indicates $L[Z|X] \neq E[Z|X]$, meaning the TSLS estimand may not admit a LATE interpretation.

In [15]:
print('='*80)
print('RAMSEY RESET TEST FOR RICH COVARIATES (H0: L[Z|X] = E[Z|X])')
print('='*80)

reset_results = []
grades_str = {'0': 'Kindergarten', '1': 'Grade 1', '2': 'Grade 2', '3': 'Grade 3'}

for g_num, g_name in grades_str.items():
    free_lunch = f'free_lunch_{g_num}'
    t_race     = f'WhiteTeacher{g_num}'
    t_exp      = f'g{g_num}tyears'
    t_md       = f'g{g_num}md'
    sch_id     = f'g{g_num}schid'

    cols  = ['initial_small', 'white_asian', 'girl', free_lunch, t_race, t_exp, t_md, sch_id]
    df_g  = df.dropna(subset=cols).copy()
    for col in cols:
        if col != sch_id:
            df_g[col] = pd.to_numeric(df_g[col], errors='coerce')
    df_g = df_g.dropna(subset=cols)

    formula = (f'initial_small ~ white_asian + girl + {free_lunch} + '
               f'{t_race} + {t_exp} + {t_md} + C({sch_id})')
    try:
        model      = smf.ols(formula, data=df_g).fit()
        reset_test = reset_ramsey(model, degree=3)
        reject     = 'Yes' if reset_test.pvalue < 0.05 else 'No'
        reset_results.append({
            'Specification / Cohort':     g_name,
            'Sample Size (N)':            len(df_g),
            'RESET F-statistic':          round(reset_test.statistic, 3),
            'p-value':                    round(reset_test.pvalue, 4),
            'Reject H0 (Rich Covariates)?': reject
        })
    except Exception as e:
        print(f'Error for {g_name}: {e}')

table_3 = pd.DataFrame(reset_results)
display(table_3)

RAMSEY RESET TEST FOR RICH COVARIATES (H0: L[Z|X] = E[Z|X])


,Specification / Cohort,Sample Size (N),RESET F-statistic,p-value,Reject H0 (Rich Covariates)?
0,Kindergarten,6282,33.671,0.0000,Yes
1,Grade 1,6810,18.538,0.0000,Yes
2,Grade 2,6739,6.255,0.0019,Yes
3,Grade 3,6736,52.171,0.0000,Yes


## 9. DDML — Partially Linear Regression ($\beta_{rich}$)

In [16]:
print('='*80)
print('DDML ESTIMATION OF SMALL CLASS EFFECTS BY GRADE')
print('='*80)

ddml_results_by_grade = {}

for grade_num, grade_name in zip(grades, grade_names):
    print(f'\n{grade_name}:')
    print('-' * 40)

    grade_data = long_df[long_df['grade'] == grade_num].copy()
    req_cols   = ['percentile', 'small_class', 'white_asian', 'girl',
                  'free_lunch', 'teacher_exp', 'teacher_masters', 'teacher_white']
    grade_data = grade_data.dropna(subset=req_cols)

    print(f'  Observations: {len(grade_data)}')
    print(f'  Small class share: {grade_data["small_class"].mean():.1%}')

    try:
        dml_grade = DoubleMLData(
            grade_data,
            y_col='percentile',
            d_cols='small_class',
            x_cols=[c for c in req_cols if c not in ['percentile', 'small_class']]
        )

        ml_g = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
        ml_m = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

        dml_plr = DoubleMLPLR(dml_grade, ml_g, ml_m, n_folds=5)
        dml_plr.fit()

        ate       = dml_plr.coef[0]
        se        = dml_plr.se[0]
        confint   = dml_plr.confint()
        ci_lower  = confint.iloc[0, 0] if not confint.empty else None
        ci_upper  = confint.iloc[0, 1] if not confint.empty else None

        ddml_results_by_grade[grade_name] = {
            'model': dml_plr, 'ate': ate, 'se': se,
            'ci_lower': ci_lower, 'ci_upper': ci_upper, 'n_obs': len(grade_data)
        }

        print(f'  ATE (DDML): {ate:.3f} (SE: {se:.3f})')
        if ci_lower is not None:
            print(f'  95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]')

    except Exception as e:
        print(f'  Error: {e}')
        ddml_results_by_grade[grade_name] = None

DDML ESTIMATION OF SMALL CLASS EFFECTS BY GRADE

Kindergarten:
----------------------------------------
  Observations: 5859
  Small class share: 30.2%
  ATE (DDML): 5.364 (SE: 0.817)
  95% CI: [3.762, 6.966]

Grade 1:
----------------------------------------
  Observations: 6622
  Small class share: 28.1%
  ATE (DDML): 5.802 (SE: 0.750)
  95% CI: [4.332, 7.273]

Grade 2:
----------------------------------------
  Observations: 6273
  Small class share: 29.7%
  ATE (DDML): 4.840 (SE: 0.756)
  95% CI: [3.357, 6.322]

Grade 3:
----------------------------------------
  Observations: 6290
  Small class share: 32.6%
  ATE (DDML): 4.828 (SE: 0.770)
  95% CI: [3.318, 6.337]


In [17]:
# Summary table
print('\n' + '='*80)
print('SUMMARY: DDML PLR ESTIMATES')
print('='*80)
print(f"{'Grade':<15} {'N':<10} {'ATE':<12} {'SE':<10} {'95% CI'}")
print('-'*75)

for _, grade_name in zip(grades, grade_names):
    if ddml_results_by_grade.get(grade_name):
        res    = ddml_results_by_grade[grade_name]
        ci_str = (f"[{res['ci_lower']:.3f}, {res['ci_upper']:.3f}]"
                  if res['ci_lower'] is not None else '[N/A]')
        print(f"{grade_name:<15} {res['n_obs']:<10} {res['ate']:<12.3f} {res['se']:<10.3f} {ci_str}")
    else:
        print(f'{grade_name:<15} {"-":<10} {"-":<12} {"-":<10} -')


SUMMARY: DDML PLR ESTIMATES
Grade           N          ATE          SE         95% CI
---------------------------------------------------------------------------
Kindergarten    5859       5.364        0.817      [3.762, 6.966]
Grade 1         6622       5.802        0.750      [4.332, 7.273]
Grade 2         6273       4.840        0.756      [3.357, 6.322]
Grade 3         6290       4.828        0.770      [3.318, 6.337]


## 10. DDML — Interactive IV Model (LATE/ACR)

In [18]:
def calculate_late_ddml(data, grade):
    """DDML IIVM — targets the LATE/ACR."""
    df_g = data[data['grade'] == grade].copy()

    covariates     = ['white_asian', 'girl', 'free_lunch',
                      'teacher_white', 'teacher_exp', 'teacher_masters']
    school_dummies = pd.get_dummies(df_g['school_id'].astype(str), prefix='sch', drop_first=True)
    df_g           = pd.concat([df_g, school_dummies], axis=1)
    x_cols         = covariates + list(school_dummies.columns)

    df_g = df_g.dropna(subset=['percentile', 'small_class', 'initial_small'] + covariates)

    dml_data = dml.DoubleMLData(
        df_g, y_col='percentile', d_cols='small_class',
        z_cols='initial_small', x_cols=x_cols
    )

    ml_g = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    ml_m = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    ml_r = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

    np.random.seed(42)
    dml_iivm = dml.DoubleMLIIVM(dml_data, ml_g, ml_m, ml_r, n_folds=5)
    dml_iivm.fit()

    coef = dml_iivm.coef[0]
    se   = dml_iivm.se[0]
    print(f'Grade {grade} — DDML LATE: {coef:.3f} ({se:.3f})')
    return coef, se


print('DDML LATE/ACR (Interactive IV Model):')
for g in [0, 1, 2, 3]:
    calculate_late_ddml(long_df, g)

DDML LATE/ACR (Interactive IV Model):
Grade 0 — DDML LATE: 4.989 (0.729)
Grade 1 — DDML LATE: 7.282 (0.795)
Grade 2 — DDML LATE: 6.641 (0.855)
Grade 3 — DDML LATE: 6.722 (0.933)


## 11. IPSW LATE

In [19]:
def calculate_late_ipsw(data, grade):
    """IPSW LATE — Hajek ratio estimator (point estimate, no SE)."""
    df_g = data[data['grade'] == grade].copy()

    covariates     = ['white_asian', 'girl', 'free_lunch',
                      'teacher_white', 'teacher_exp', 'teacher_masters']
    school_dummies = pd.get_dummies(df_g['school_id'].astype(str), prefix='sch', drop_first=True)
    df_g           = pd.concat([df_g, school_dummies], axis=1)
    x_cols         = covariates + list(school_dummies.columns)

    df_g = df_g.dropna(subset=['percentile', 'small_class', 'initial_small'] + covariates)

    Y = df_g['percentile'].values
    T = df_g['small_class'].values
    Z = df_g['initial_small'].values
    X = sm.add_constant(df_g[x_cols].astype(float))

    # Instrument propensity score E[Z|X]
    logit_model = sm.Logit(Z, X).fit(disp=0)
    p_X         = np.clip(logit_model.predict(X), 0.01, 0.99)

    # Normalised Hajek weights
    w1 = Z / p_X;             w0 = (1 - Z) / (1 - p_X)
    w1_norm = w1 / np.sum(w1); w0_norm = w0 / np.sum(w0)

    itt_Y = np.sum(w1_norm * Y) - np.sum(w0_norm * Y)
    itt_T = np.sum(w1_norm * T) - np.sum(w0_norm * T)

    beta_ipsw = itt_Y / itt_T
    print(f'Grade {grade} — IPSW LATE: {beta_ipsw:.3f}')
    return beta_ipsw


print('IPSW LATE (point estimates):')
for g in [0, 1, 2, 3]:
    calculate_late_ipsw(long_df, g)

IPSW LATE (point estimates):
Grade 0 — IPSW LATE: 5.529
Grade 1 — IPSW LATE: 7.234
Grade 2 — IPSW LATE: 6.655
Grade 3 — IPSW LATE: 7.459
